In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
Experiment 2: Causal ZK Metric for Communication Verification
==============================================================

The problem with existing metrics (Lowe et al.'s causal influence of
communication):
    They require you to know something about message *content* to verify
    communication is happening. You decode messages and check alignment
    with concepts.

Our claim:
    You can verify communication is real WITHOUT knowing what is being
    communicated. Like a zero-knowledge proof — we prove the fact
    (communication exists) without revealing the witness (message content).

Operationalization:
    1. CAUSAL DELTA: Run forward pass with real messages. Then scramble
       messages (replace with random from same distribution). Measure
       delta in receiver's output distribution: KL(real || scrambled).
       High delta = receiver is actually using the message.

    2. RECURSIVE COUPLING: In a multi-turn setup, check whether the
       receiver's *changed* behavior (caused by scrambled message) in
       turn T feeds back to change the sender's message in turn T+1.
       If yes: there is recursive causal coupling — a hallmark of genuine
       communication, not just one-way signal reading.

    3. ZK SCORE = CIC * RC_coefficient
       where CIC = causal influence of communication (step 1)
             RC  = recursive coupling strength (step 2)

    Neither step requires decoding message content.

Setup:
    We build a simple multi-turn referential game:
      - Turn 1: sender sees object, sends message M1
      - Turn 1: receiver gets M1, sends back a "query" Q1
      - Turn 2: sender sees Q1, sends refined message M2
      - Turn 2: receiver gets M2, makes final prediction
    The recursion is: does scrambling M1 change Q1, and does changed
    Q1 change M2? If yes, there is recursive causal coupling.

Kaggle setup:
    !git clone https://github.com/facebookresearch/EGG.git
    !pip install -e EGG
    !pip install scipy matplotlib seaborn
"""

import os
import json
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def make_dataset(n_features, n_distractors, n_samples, seed=42):
    rng = np.random.default_rng(seed)
    objects = rng.integers(0, 2,
        size=(n_samples, n_distractors + 1, n_features)).astype(np.float32)
    sender_input   = torch.tensor(objects[:, 0, :])
    labels         = torch.zeros(n_samples, dtype=torch.long)
    receiver_input = torch.tensor(objects)
    return TensorDataset(sender_input, labels, receiver_input)


# ---------------------------------------------------------------------------
# Agents: multi-turn capable
# ---------------------------------------------------------------------------

class MultiTurnSender(nn.Module):
    """
    Sender that can condition on receiver's query as well as the object.
    Turn 1: only sees object.
    Turn 2: sees object + receiver's query vector.
    """
    def __init__(self, n_features, hidden, vocab_size, query_dim):
        super().__init__()
        self.turn1 = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Linear(hidden, vocab_size)
        )
        self.turn2 = nn.Sequential(
            nn.Linear(n_features + query_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, vocab_size)
        )

    def forward_turn1(self, obj):
        return self.turn1(obj)   # logits over vocab

    def forward_turn2(self, obj, query):
        return self.turn2(torch.cat([obj, query], dim=-1))


class MultiTurnReceiver(nn.Module):
    """
    Receiver that:
      - After turn 1 message: emits a query vector (asking for more info)
      - After turn 2 message: makes a final prediction over candidates
    """
    def __init__(self, n_features, hidden, msg_dim, n_distractors, query_dim):
        super().__init__()
        self.n_candidates = n_distractors + 1
        self.msg_embed   = nn.Linear(msg_dim, hidden)
        self.obj_embed   = nn.Linear(n_features, hidden)
        # query: what the receiver asks after seeing turn 1 message
        self.query_head  = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, query_dim), nn.Tanh()
        )
        # final prediction head (used after turn 2)
        # takes turn1 msg + turn2 msg + object representations
        self.predict_head = nn.Linear(hidden * 2, self.n_candidates)

    def forward_turn1(self, msg, receiver_input):
        """Returns query vector."""
        msg_h = self.msg_embed(msg)          # (B, hidden)
        return self.query_head(msg_h)        # (B, query_dim)

    def forward_turn2(self, msg1, msg2, receiver_input):
        """Returns scores over candidates."""
        h1 = self.msg_embed(msg1)                      # (B, hidden)
        h2 = self.msg_embed(msg2)                      # (B, hidden)
        combined = torch.cat([h1, h2], dim=-1)         # (B, 2*hidden)
        return self.predict_head(combined)             # (B, n_candidates)


class MultiTurnGame(nn.Module):
    """
    Wraps two-turn communication.  Messages are discrete (argmax + one-hot)
    during evaluation; during training we use Gumbel-Softmax straight-through
    so gradients flow.
    """
    def __init__(self, sender, receiver, vocab_size,
                 gs_temperature=1.0, hard=True):
        super().__init__()
        self.sender      = sender
        self.receiver    = receiver
        self.vocab_size  = vocab_size
        self.temperature = gs_temperature
        self.hard        = hard

    def _discretize(self, logits):
        """Gumbel-Softmax straight-through during training, argmax at eval."""
        if self.training:
            return F.gumbel_softmax(logits, tau=self.temperature, hard=self.hard)
        else:
            idx = logits.argmax(dim=-1)
            return F.one_hot(idx, self.vocab_size).float()

    def forward(self, sender_input, labels, receiver_input):
        # --- Turn 1 ---
        logits1 = self.sender.forward_turn1(sender_input)
        msg1    = self._discretize(logits1)

        query   = self.receiver.forward_turn1(msg1, receiver_input)

        # --- Turn 2 ---
        logits2 = self.sender.forward_turn2(sender_input, query)
        msg2    = self._discretize(logits2)

        scores  = self.receiver.forward_turn2(msg1, msg2, receiver_input)

        loss = F.cross_entropy(scores, labels)
        acc  = (scores.argmax(-1) == labels).float().mean()
        return loss, {"acc": acc}, msg1, msg2, query, scores


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

def train(args, train_ds, val_ds):
    vocab_size = args.vocab_size
    query_dim  = args.query_dim
    hidden     = args.hidden

    sender   = MultiTurnSender(args.n_features, hidden, vocab_size, query_dim)
    receiver = MultiTurnReceiver(args.n_features, hidden, vocab_size,
                                 args.n_distractors, query_dim)
    game     = MultiTurnGame(sender, receiver, vocab_size,
                             gs_temperature=args.gs_temp)

    optimizer = torch.optim.Adam(game.parameters(), lr=args.lr)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=256)

    print("Training multi-turn game...")
    for epoch in range(args.n_epochs):
        game.train()
        for si, lb, ri in train_loader:
            optimizer.zero_grad()
            loss, info, *_ = game(si, lb, ri)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            game.eval()
            val_accs = []
            with torch.no_grad():
                for si, lb, ri in val_loader:
                    _, info, *_ = game(si, lb, ri)
                    val_accs.append(info["acc"].item())
            print(json.dumps({"epoch": epoch+1,
                              "val_acc": round(float(np.mean(val_accs)), 4)}))

    return game


# ---------------------------------------------------------------------------
# ZK Metric
# ---------------------------------------------------------------------------

def compute_zk_metric(game, dataset, args, n_scrambles=10):
    """
    Compute the ZK communication score.

    Returns a dict with:
        cic_turn1     : causal influence of message 1 on query
        cic_turn2     : causal influence of message 2 on final scores
        rc_coeff      : recursive coupling coefficient
        zk_score      : cic_turn2 * rc_coeff  (the headline number)

    All computed WITHOUT inspecting message content.
    """
    game.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)

    cic1_vals, cic2_vals, rc_vals = [], [], []

    with torch.no_grad():
        for si, lb, ri in loader:
            B = si.size(0)

            # ---- Real forward pass ----
            _, _, msg1_real, msg2_real, query_real, scores_real = game(si, lb, ri)
            probs_real = F.softmax(scores_real, dim=-1)

            # ---- Scramble M1, measure effect on query (CIC turn 1) ----
            cic1_batch = []
            rc_batch   = []
            for _ in range(n_scrambles):
                # random message from same vocab (permute within batch)
                perm       = torch.randperm(B)
                msg1_fake  = msg1_real[perm]          # scrambled: same distribution, wrong content

                query_fake = game.receiver.forward_turn1(msg1_fake, ri)

                # CIC1: how much does query change? (KL between real and fake query distributions)
                # We treat query as a Gaussian and use MSE as a proxy for KL
                cic1 = F.mse_loss(query_fake, query_real, reduction='none').mean(dim=-1)
                cic1_batch.append(cic1)

                # Recursive coupling: does changed query → changed M2?
                logits2_fake = game.sender.forward_turn2(si, query_fake)
                msg2_fake    = (logits2_fake.argmax(-1) == msg2_real.argmax(-1)).float()
                # RC = fraction of M2 tokens that CHANGED when query changed
                rc = 1.0 - msg2_fake.mean()
                rc_batch.append(rc.item())

            cic1_mean = torch.stack(cic1_batch).mean(dim=0)  # (B,)
            cic1_vals.extend(cic1_mean.tolist())
            rc_vals.append(float(np.mean(rc_batch)))

            # ---- Scramble M2, measure effect on final scores (CIC turn 2) ----
            cic2_batch = []
            for _ in range(n_scrambles):
                perm      = torch.randperm(B)
                msg2_fake = msg2_real[perm]

                scores_fake = game.receiver.forward_turn2(msg1_real, msg2_fake, ri)
                probs_fake  = F.softmax(scores_fake, dim=-1)

                # KL(real || fake) per sample
                kl = F.kl_div(probs_fake.log(), probs_real, reduction='none').sum(-1)
                cic2_batch.append(kl)

            cic2_mean = torch.stack(cic2_batch).mean(dim=0)
            cic2_vals.extend(cic2_mean.tolist())

    cic1   = float(np.mean(cic1_vals))
    cic2   = float(np.mean(cic2_vals))
    rc     = float(np.mean(rc_vals))
    zk     = cic2 * rc

    return {
        "cic_turn1"  : round(cic1, 5),
        "cic_turn2"  : round(cic2, 5),
        "rc_coeff"   : round(rc,   5),
        "zk_score"   : round(zk,   5),
    }


# ---------------------------------------------------------------------------
# Ablation: ZK score on a non-communicating game (baseline)
# ---------------------------------------------------------------------------

def make_baseline_game(args):
    """
    A game where the channel is severed: receiver always gets a *fixed*
    zero vector as message regardless of what sender produces.
    ZK score should be ~0 here.
    """
    vocab_size = args.vocab_size
    query_dim  = args.query_dim
    hidden     = args.hidden

    sender   = MultiTurnSender(args.n_features, hidden, vocab_size, query_dim)
    receiver = MultiTurnReceiver(args.n_features, hidden, vocab_size,
                                 args.n_distractors, query_dim)

    class SeveredGame(nn.Module):
        def __init__(self, sender, receiver, vocab_size):
            super().__init__()
            self.sender    = sender
            self.receiver  = receiver
            self.vocab_size = vocab_size

        def forward(self, sender_input, labels, receiver_input):
            B = sender_input.size(0)
            # sender produces message but receiver never sees it
            logits1 = self.sender.forward_turn1(sender_input)
            msg1    = F.one_hot(logits1.argmax(-1), self.vocab_size).float()

            # receiver gets ZERO message regardless
            zero_msg = torch.zeros_like(msg1)
            query    = self.receiver.forward_turn1(zero_msg, receiver_input)

            logits2  = self.sender.forward_turn2(sender_input, query)
            msg2     = F.one_hot(logits2.argmax(-1), self.vocab_size).float()
            scores   = self.receiver.forward_turn2(zero_msg, zero_msg, receiver_input)

            loss = F.cross_entropy(scores, labels)
            acc  = (scores.argmax(-1) == labels).float().mean()
            return loss, {"acc": acc}, msg1, msg2, query, scores

    game = SeveredGame(sender, receiver, vocab_size)
    optimizer = torch.optim.Adam(game.parameters(), lr=args.lr)

    train_ds = make_dataset(args.n_features, args.n_distractors, 2000, seed=99)
    for epoch in range(args.n_epochs):
        game.train()
        for si, lb, ri in DataLoader(train_ds, batch_size=args.batch_size, shuffle=True):
            optimizer.zero_grad()
            loss, *_ = game(si, lb, ri)
            loss.backward()
            optimizer.step()

    return game


# ---------------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------------

def plot_zk_comparison(real_scores, baseline_scores, save_path):
    fig, ax = plt.subplots(figsize=(7, 5))
    labels   = ["Communicating\ngame", "Severed channel\n(baseline)"]
    zk_vals  = [real_scores["zk_score"], baseline_scores["zk_score"]]
    cic_vals = [real_scores["cic_turn2"], baseline_scores["cic_turn2"]]
    rc_vals  = [real_scores["rc_coeff"],  baseline_scores["rc_coeff"]]

    x = np.arange(2)
    w = 0.25
    ax.bar(x - w, zk_vals,  width=w, label="ZK score",   color="#2196F3")
    ax.bar(x,     cic_vals, width=w, label="CIC turn 2", color="#FF9800")
    ax.bar(x + w, rc_vals,  width=w, label="RC coeff",   color="#4CAF50")

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("ZK Communication Metric:\nCommunicating vs Severed Game", fontsize=13)
    ax.legend()
    ax.set_ylim(0, max(cic_vals) * 1.3 + 0.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Plot saved to {save_path}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--n_features",    type=int,   default=16)
    p.add_argument("--n_distractors", type=int,   default=4)
    p.add_argument("--n_train",       type=int,   default=8000)
    p.add_argument("--n_val",         type=int,   default=2000)
    p.add_argument("--vocab_size",    type=int,   default=16)
    p.add_argument("--query_dim",     type=int,   default=16)
    p.add_argument("--hidden",        type=int,   default=64)
    p.add_argument("--n_epochs",      type=int,   default=60)
    p.add_argument("--batch_size",    type=int,   default=128)
    p.add_argument("--lr",            type=float, default=1e-3)
    p.add_argument("--gs_temp",       type=float, default=0.5)
    p.add_argument("--n_scrambles",   type=int,   default=20)
    p.add_argument("--results_dir",   type=str,   default="results")
    p.add_argument("--seed",          type=int,   default=42)
    return p.parse_args()


def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    os.makedirs(args.results_dir, exist_ok=True)

    train_ds = make_dataset(args.n_features, args.n_distractors, args.n_train, args.seed)
    val_ds   = make_dataset(args.n_features, args.n_distractors, args.n_val,   args.seed + 1)

    # --- Train real communicating game ---
    print("\n=== Training communicating game ===")
    game = train(args, train_ds, val_ds)

    print("\n=== Training severed-channel baseline ===")
    baseline_game = make_baseline_game(args)

    # --- Compute ZK metric on both ---
    print("\n=== Computing ZK metric ===")
    real_scores     = compute_zk_metric(game,          val_ds, args, args.n_scrambles)
    baseline_scores = compute_zk_metric(baseline_game, val_ds, args, args.n_scrambles)

    print("\nCommunicating game:")
    print(json.dumps(real_scores, indent=2))
    print("\nSevered baseline:")
    print(json.dumps(baseline_scores, indent=2))

    results = {"communicating": real_scores, "severed_baseline": baseline_scores}
    with open(os.path.join(args.results_dir, "zk_results.json"), "w") as f:
        json.dump(results, f, indent=2)

    plot_zk_comparison(
        real_scores, baseline_scores,
        os.path.join(args.results_dir, "zk_comparison.png")
    )

    # Summary interpretation
    ratio = real_scores["zk_score"] / (baseline_scores["zk_score"] + 1e-9)
    print(f"\nZK ratio (communicating / baseline): {ratio:.2f}x")
    print("A ratio >> 1 confirms genuine recursive causal communication.")


if __name__ == "__main__":
    main()